# Python Data Science Interviews — Machine and Deep Learning

Run all cells in a fresh Python 3.11 CPU kernel. The original placeholder dataset is completed with explicit examples. The transfer-learning demonstration uses real bundled handwritten digits and actual pretrained ImageNet VGG16 weights; it is not a large-scale benchmark. Original source code remains in a nonexecuting appendix.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['KERAS_BACKEND'] = 'tensorflow'
import sys
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', 'numpy==2.3.5', 'pandas==2.3.3', 'scikit-learn==1.7.2', 'tensorflow==2.20.0', 'keras==3.15.1', 'matplotlib==3.10.7', 'pillow==12.0.0', 'certifi==2026.10.23'], check=True)
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib
import numpy as np
import tensorflow as tf
import keras
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
import matplotlib.pyplot as plt
get_ipython().run_line_magic('matplotlib', 'inline')
np.set_printoptions(precision=6, suppress=True)
work = TemporaryDirectory(prefix='python-ml-interview-')
artifact_dir = Path(work.name)

def batched(features, labels, batch_size=32):
    options = tf.data.Options()
    options.threading.private_threadpool_size = 1
    options.threading.max_intra_op_parallelism = 1
    return tf.data.Dataset.from_tensor_slices((features, labels)).batch(batch_size).with_options(options)

print('Python:', sys.version.split()[0], '| TensorFlow:', tf.__version__, '| Keras:', keras.__version__)

In [ ]:
from sklearn.model_selection import train_test_split
print("train_test_split imported.")

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
iris = load_iris()
X, y = iris.data, iris.target
row_ids = np.arange(len(y))
clf = DecisionTreeClassifier(random_state=42)
print('Iris features/labels:', X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_ids, test_ids = train_test_split(row_ids, test_size=.2, random_state=42)
print("Training/test rows:", len(X_train), len(X_test))

In [ ]:
clf.fit(X_train, y_train)
iris_classifier = clf
print("Learned classes:", clf.classes_.tolist())

In [ ]:
accuracy = clf.score(X_test, y_test)
print("Held-out Iris accuracy:", accuracy)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Example: Generating some synthetic data
# In a real-world scenario, you would load your data here
X, y = [[1, 2], [2, 3], [3, 4], [4, 5]], [0, 0, 1, 1]

# Step 2: Split the Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Step 3: Instantiate the Model
clf = DecisionTreeClassifier(random_state=42)

# Step 4: Fit the Model
clf.fit(X_train, y_train)

# Step 5: Make Predictions
predictions = clf.predict(X_test)

# Step 6: Evaluate the Model
accuracy = accuracy_score(y_test, predictions)

print(f"Model Accuracy: {accuracy}")
toy_tree_accuracy = accuracy

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Example data (replace this with your dataset)
X, y = [[1, 2], [2, 3], [3, 4], [4, 5]], [0, 0, 1, 1]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Create a pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Step 1: Scale features
    ('classifier', LogisticRegression())  # Step 2: Apply the classifier
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the test data
predictions = pipeline.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy}")
toy_pipeline = pipeline
toy_pipeline_X_train = np.asarray(X_train).copy()
toy_pipeline_accuracy = accuracy

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)
X, y = rng.random((1000, 10), dtype=np.float32), rng.integers(0, 2, (1000, 1)).astype(np.float32)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=.2, random_state=42)
callback_model = Sequential([Input(shape=(10,)), Dense(64, activation='relu'), Dense(1, activation='sigmoid')])
callback_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint_path = artifact_dir / 'best_model.keras'
model_checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True)
history = callback_model.fit(batched(X_train, y_train), validation_data=batched(X_val, y_val), epochs=20, callbacks=[early_stopping, model_checkpoint], shuffle=False, verbose=0)
print('Random-label training rows / validation rows:', len(X_train), len(X_val))
print('Completed epochs:', len(history.history['loss']))
print('Best validation loss:', round(min(history.history['val_loss']), 6))
print('Best model checkpoint saved:', checkpoint_path.exists())
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('Binary cross-entropy'); plt.legend(); plt.title('Callback training example')
plt.show()

In [ ]:
from sklearn.datasets import load_digits
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.optimizers import Adam

tf.keras.utils.set_random_seed(42)
digits = load_digits()
# A small real, ten-class image dataset makes the CPU example self-contained.
selected_ids = np.concatenate([np.flatnonzero(digits.target == label)[:12] for label in range(10)])
train_image_ids, val_image_ids = train_test_split(selected_ids, test_size=20, random_state=42, stratify=digits.target[selected_ids])
def prepare_images(indices):
    gray = digits.images[indices].astype(np.float32) * (255.0 / 16.0)
    rgb = np.repeat(gray[..., None], 3, axis=-1)
    resized = tf.image.resize(rgb, (224, 224)).numpy()
    return preprocess_input(resized)
train_images, val_images = prepare_images(train_image_ids), prepare_images(val_image_ids)
train_labels = tf.keras.utils.to_categorical(digits.target[train_image_ids], 10)
val_labels = tf.keras.utils.to_categorical(digits.target[val_image_ids], 10)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False
backbone_before = hashlib.sha256(b''.join(weight.numpy().tobytes() for weight in base_model.weights)).hexdigest()
def frozen_features(images):
    return np.concatenate([base_model(images[i:i+10], training=False).numpy() for i in range(0, len(images), 10)])
train_features, val_features = frozen_features(train_images), frozen_features(val_images)
# Frozen features are cached once; the original Flatten/Dense(256)/Dense(10) head trains for ten epochs.
head = Sequential([Input(shape=base_model.output_shape[1:]), Flatten(), Dense(256, activation='relu'), Dense(10, activation='softmax')])
head.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
head_before = [weight.copy() for weight in head.get_weights()]
transfer_history = head.fit(batched(train_features, train_labels), validation_data=batched(val_features, val_labels), epochs=10, shuffle=False, verbose=0)
transfer_model = Sequential([Input(shape=(224, 224, 3)), base_model, head])
transfer_predictions = transfer_model(val_images[:3], training=False).numpy()
print('Real digit images: training', len(train_image_ids), '| validation', len(val_image_ids), '| classes', train_labels.shape[1])
print('Frozen feature shape:', train_features.shape)
print('Head epochs:', len(transfer_history.history['loss']))
print('Final validation accuracy:', round(transfer_history.history['val_accuracy'][-1], 6))
print('Three prediction row sums:', np.round(transfer_predictions.sum(axis=1), 6).tolist())

In [ ]:
sequential_model = Sequential([Input(shape=(100,)), Dense(64, activation='relu'), Dense(10, activation='softmax')])
sequential_output = sequential_model(np.zeros((2, 100), dtype=np.float32), training=False).numpy()
print('Sequential output:', sequential_output.shape)

In [ ]:
from tensorflow.keras.models import Model
input_layer = Input(shape=(100,))
hidden_layer = Dense(64, activation='relu')(input_layer)
output_layer = Dense(10, activation='softmax')(hidden_layer)
functional_model = Model(inputs=input_layer, outputs=output_layer)
functional_output = functional_model(np.zeros((2, 100), dtype=np.float32), training=False).numpy()
print('Functional output:', functional_output.shape)

In [ ]:
sequential_block = Sequential([Input(shape=(100,)), Dense(64, activation='relu')])
input_layer = Input(shape=(100,))
hidden_layer = sequential_block(input_layer)
output_layer = Dense(10, activation='softmax')(hidden_layer)
combined_model = Model(inputs=input_layer, outputs=output_layer)
combined_output = combined_model(np.zeros((2, 100), dtype=np.float32), training=False).numpy()
print('Hybrid output:', combined_output.shape)
print('Sequential block is connected:', sequential_block in combined_model.layers)

In [ ]:
from tensorflow.keras.optimizers import Adam
compiled_model = Sequential([Input(shape=(10,)), Dense(64, activation='relu'), Dense(1, activation='sigmoid')])
compiled_model.compile(optimizer=Adam(learning_rate=.001), loss='binary_crossentropy', metrics=['accuracy'])
batch_result = compiled_model.train_on_batch(np.ones((4, 10), dtype=np.float32), np.array([[0], [1], [0], [1]], dtype=np.float32), return_dict=True)
print('Compiled model trained one batch:', {key: round(float(value), 6) for key, value in batch_result.items()})

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
datagen = ImageDataGenerator(rotation_range=20, width_shift_range=.2, height_shift_range=.2, shear_range=.2, zoom_range=.2, horizontal_flip=True, fill_mode='nearest')
def load_and_preprocess_image():
    # Real bundled digit image, resized as RGB without VGG-specific channel conversion.
    gray = digits.images[0].astype(np.float32) * (255.0 / 16.0)
    return tf.image.resize(np.repeat(gray[..., None], 3, axis=-1), (224, 224)).numpy()
image = load_and_preprocess_image()
augmented_images = [datagen.random_transform(image.copy(), seed=seed) for seed in range(4)]
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for axis, frame, title in zip(axes, [image] + augmented_images, ['Original', 'Augment 1', 'Augment 2', 'Augment 3', 'Augment 4']):
    axis.imshow(np.clip(frame / 255., 0., 1.)); axis.set_title(title); axis.axis('off')
fig.tight_layout(); plt.show()
print('Augmented images:', len(augmented_images), '| shape:', augmented_images[0].shape)

In [ ]:
from sklearn.metrics import accuracy_score
assert set(train_ids).isdisjoint(test_ids) and len(train_ids) + len(test_ids) == 150
assert iris_classifier.score(iris.data[test_ids], iris.target[test_ids]) == accuracy_score(iris.target[test_ids], iris_classifier.predict(iris.data[test_ids]))
np.testing.assert_allclose(toy_pipeline.named_steps['scaler'].mean_, toy_pipeline_X_train.mean(axis=0))
assert 0 <= toy_tree_accuracy <= 1 and 0 <= toy_pipeline_accuracy <= 1
assert 1 <= len(history.history['loss']) <= 20
assert all(np.isfinite(value).all() for value in history.history.values())
assert checkpoint_path.exists()
loaded_best = tf.keras.models.load_model(checkpoint_path)
np.testing.assert_allclose(callback_model(X_val[:10], training=False).numpy(), loaded_best(X_val[:10], training=False).numpy(), atol=1e-6)
# Exercise callback patience explicitly without inventing a training-loss history.
probe = Sequential([Input(shape=(1,)), Dense(1)])
probe.stop_training = False
callback_probe = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
callback_probe.set_model(probe); callback_probe.on_train_begin()
for epoch, value in enumerate([.5, .4, .45, .46, .47]):
    callback_probe.on_epoch_end(epoch, {'val_loss': value})
assert probe.stop_training and callback_probe.stopped_epoch == 4
assert set(train_image_ids).isdisjoint(val_image_ids)
assert len(train_image_ids) == 100 and len(val_image_ids) == 20
assert set(digits.target[train_image_ids]) == set(range(10))
assert set(digits.target[val_image_ids]) == set(range(10))
assert train_features.shape == (100, 7, 7, 512) and len(transfer_history.history['loss']) == 10
assert not base_model.trainable and not base_model.trainable_weights
backbone_after = hashlib.sha256(b''.join(weight.numpy().tobytes() for weight in base_model.weights)).hexdigest()
assert backbone_before == backbone_after
assert any(not np.array_equal(before, after) for before, after in zip(head_before, head.get_weights()))
np.testing.assert_allclose(transfer_predictions, head(val_features[:3], training=False).numpy(), atol=1e-5)
np.testing.assert_allclose(transfer_predictions.sum(axis=1), 1, atol=1e-6)
pixel = preprocess_input(np.array([[[[10., 20., 30.]]]], dtype=np.float32))
np.testing.assert_allclose(pixel, [[[[30.-103.939, 20.-116.779, 10.-123.68]]]], atol=1e-5)
for output in (sequential_output, functional_output, combined_output):
    assert output.shape == (2, 10) and np.isfinite(output).all()
    np.testing.assert_allclose(output.sum(axis=1), 1, atol=1e-6)
assert sequential_block in combined_model.layers
assert all(np.isfinite(value) for value in batch_result.values())
for seed, augmented in enumerate(augmented_images):
    assert augmented.shape == image.shape and np.isfinite(augmented).all()
    assert not np.array_equal(augmented, image)
    np.testing.assert_allclose(augmented, datagen.random_transform(image.copy(), seed=seed))
assert not plt.get_fignums()
print('Real train/test separation, sklearn scaling statistics, callback training/checkpoint reload and patience behavior passed.')
print('ImageNet VGG16 weights unchanged; head weights updated for ten epochs; full-model/cached-feature predictions and independent RGB-to-BGR mean subtraction match.')
print('Sequential/functional/hybrid forward passes, compiled-model batch training and four real seeded augmentations passed.')
work.cleanup()
print('Temporary checkpoint artifacts cleaned up.')

## Original source code appendix

Unmodified archive blocks, not executed.

### Original block 1

```python
from sklearn.model_selection import train_test_split
```

### Original block 2

```python
# Example: Assuming X is the feature matrix and y is the target variable
```

### Original block 3

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

### Original block 4

```python
# Example: Assuming clf is the machine learning model clf.fit(X_train, y_train)
```

### Original block 5

```python
# Example: Assuming clf is the trained model accuracy = clf.score(X_test, y_test)
```

### Original block 6

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Example: Generating some synthetic data
# In a real-world scenario, you would load your data here
X, y = [[1, 2], [2, 3], [3, 4], [4, 5]], [0, 0, 1, 1]

# Step 2: Split the Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Step 3: Instantiate the Model
clf = DecisionTreeClassifier()

# Step 4: Fit the Model
clf.fit(X_train, y_train)

# Step 5: Make Predictions
predictions = clf.predict(X_test)

# Step 6: Evaluate the Model
accuracy = accuracy_score(y_test, predictions)

print(f"Model Accuracy: {accuracy}")
```

### Original block 7

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Example data (replace this with your dataset)
X, y = [[1, 2], [2, 3], [3, 4], [4, 5]], [0, 0, 1, 1]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Create a pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Step 1: Scale features
    ('classifier', LogisticRegression())  # Step 2: Apply the classifier
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the test data
predictions = pipeline.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy}")
```

### Original block 8

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
import numpy as np

# Generate synthetic data (replace with your dataset)
X, y = np.random.rand(1000, 10), np.random.randint(2, size=(1000, 1))

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Build a simple neural network model
model = Sequential()
model.add(Dense(64, input_dim=10, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True)

# Train the model with callbacks
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, callbacks=[early_stopping, model_checkpoint])
```

### Original block 9

```python
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load pre-trained VGG16 model (excluding top layers)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the pre-trained layers
for layer in base_model.layers:
    layer.trainable = False

# Create a new model and add VGG16 base model
model = Sequential()
model.add(base_model)

# Add new layers for classification
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dense(10, activation='softmax'))  # Replace with the number of classes in the new task

# Compile the model
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Load and preprocess data (replace with your data loading code)
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'path/to/train_data',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    'path/to/test_data',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Train the model
model.fit(train_generator, epochs=10, validation_data=test_generator)
```

### Original block 10

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()
model.add(Dense(64, input_dim=100, activation='relu'))
model.add(Dense(10, activation='softmax'))
```

### Original block 11

```python
from tensorflow.keras.layers import Input, Dense, concatenate
from tensorflow.keras.models import Model

input_layer = Input(shape=(100,))
hidden_layer = Dense(64, activation='relu')(input_layer)
output_layer = Dense(10, activation='softmax')(hidden_layer)

model = Model(inputs=input_layer, outputs=output_layer)
```

### Original block 12

```python
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Input

# Sequential API for the simple part
sequential_model = Sequential()
sequential_model.add(Dense(64, input_dim=100, activation='relu'))

# Functional API for the complex part
input_layer = Input(shape=(100,))
hidden_layer = Dense(64, activation='relu')(input_layer)
output_layer = Dense(10, activation='softmax')(hidden_layer)

# Combine both parts
combined_model = Model(inputs=input_layer, outputs=output_layer)
```

### Original block 13

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()
model.add(Dense(64, input_dim=10, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
```

### Original block 14

```python
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create an instance of ImageDataGenerator with specified augmentations
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Load and preprocess an example image
# Apply data augmentation on the fly during training
image = load_and_preprocess_image()  # Replace with your image loading and preprocessing code
augmented_images = [datagen.random_transform(image) for _ in range(4)]  # Generate 4 augmented images
```